# FirstAI / Orbit — conversational Colab T4 training
Build a larger FirstGPT from random initialization, add multi-turn assistant conversations, train on sourced text and practical tasks, inspect actual sample replies, then download the checkpoint. No Drive mount; Colab storage is temporary.


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO=Path('/content/FirstAI')
if REPO.exists(): subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','https://github.com/llakshyaa-rgb/FirstAI.git',str(REPO)],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, reconnect, and rerun.')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
subprocess.run([sys.executable,'-m','pip','install','-q','datasets>=3.0'],check=True)


## 1. Prepare base text, task data and multi-turn conversations
Downloads up to 30M WikiText characters, capability tasks, sourced public-data examples and up to 20,000 assistant turns from UltraChat 200k (training split only). Conversation rows preserve preceding turns so SFT teaches dialogue continuation, not just isolated one-shot instructions.


In [ ]:
subprocess.run([sys.executable,'build_all_data.py','--country','IND','--city','Patna','--max-per-dataset','10000','--max-base-chars','30000000','--per-source-cap','20000','--conversation-limit','20000'],check=True)
import json
for f in ['data/capabilities/manifest.json','data/instructions_conversations.manifest.json','data/instructions_balanced.manifest.json']:
 p=Path(f)
 if p.exists(): print('\n'+f, json.dumps(json.loads(p.read_text()),indent=2)[:12000])


## 2. Train a substantially larger FirstGPT from scratch
This profile uses 8 layers, width 384, 6 attention heads, a 2,048-token BPE vocabulary and 1,024-token context (~15M parameters, depending on the learned vocabulary size). It is still a small experimental model, not a frontier model. T4 GPU is required; 12,000 steps may take hours and Colab session limits are not guaranteed.


In [ ]:
base_args=['--data','data/input.txt','--out','checkpoints/firstgpt-convo.pt','--steps',os.environ.get('FIRSTAI_BASE_STEPS','12000'),'--eval-interval','200','--checkpoint-interval','500','--eval-batches','8','--batch-size','2','--grad-accum-steps','8','--block-size','1024','--layers','8','--heads','6','--width','384','--vocab-size','2048','--tokenizer-train-chars','100000','--architecture-version','2','--amp']
if Path('checkpoints/firstgpt-convo.pt').exists() and os.environ.get('FIRSTAI_RESTART_BASE','0')!='1': base_args.append('--resume')
subprocess.run([sys.executable,'train.py',*base_args],check=True)


## 3. Supervised conversational instruction tuning
This stage mixes multi-turn assistant conversations with sourced mathematics, science, reading, coding and Orbit behaviour examples. It trains only on assistant response tokens; user prompts are context, not target text. Resume is allowed only when the dataset hash matches.


In [ ]:
sft_args=['--checkpoint','checkpoints/firstgpt-convo.best.pt','--data','data/instructions_balanced.jsonl','--out','checkpoints/orbit-convo-instruct.pt','--steps',os.environ.get('FIRSTAI_SFT_STEPS','6000'),'--batch-size','4','--eval-interval','100','--lr','0.00003']
if Path('checkpoints/orbit-convo-instruct.pt').exists() and os.environ.get('FIRSTAI_RESTART_SFT','0')!='1': sft_args.append('--resume')
subprocess.run([sys.executable,'finetune_instructions.py',*sft_args],check=True)
print('Best SFT checkpoint:',Path('checkpoints/orbit-convo-instruct.best.pt').stat().st_size,'bytes')


## 4. Inspect real replies before deploying
This is a qualitative smoke test, not an automatic proof of intelligence. Read the answers: if they loop, ignore context or invent facts, do not deploy the checkpoint as ready.


In [ ]:
from orbit.runtime import OrbitRuntime
runtime=OrbitRuntime('checkpoints/orbit-convo-instruct.best.pt',max_new_tokens=100)
history=[]
for prompt in ['My project is called Nebula.','What did I call my project?','And what is it called?']:
 answer=runtime.reply(prompt,history=history)
 print('\nOWNER:',prompt,'\nORBIT:',answer)
 history.extend([{'role':'user','content':prompt},{'role':'assistant','content':answer}])
print('\nGENERAL TEST:',runtime.reply('Explain why the sky looks blue in simple terms.'))


## 5. Download the trained checkpoints
Download both files before disconnecting; Colab's temporary disk is erased when the runtime ends. Use the SFT best checkpoint for Orbit.


In [ ]:
from google.colab import files
files.download('checkpoints/firstgpt-convo.best.pt')
files.download('checkpoints/orbit-convo-instruct.best.pt')
